In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


After mounting your Google Drive, you can access your files. A common path for your files will be `/content/drive/My Drive/`. You'll need to specify the full path to your data file. If you're unsure of the path, you can use the file browser icon on the left sidebar to navigate your Drive and copy the path to your file.

In [6]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트 (필요 시 주석 해제)
# drive.mount('/content/drive')

# 2. 질병 데이터 로드 및 기본 전처리 (Wide -> Long)
disease_path = '/content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/merged_zzzzgamusi_data.csv'

print("질병 데이터 로드를 시작합니다...")
try:
    df_disease = pd.read_csv(disease_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_disease = pd.read_csv(disease_path, encoding='cp949')

if '합계' in df_disease.columns:
    df_disease = df_disease.drop(columns=['합계'])

# 가로 데이터를 세로 데이터로 변환 (전체 지역 포함)
df_disease_long = df_disease.melt(id_vars=['지역', '연도'], var_name='주차', value_name='환자수')
df_disease_long['주차'] = df_disease_long['주차'].str.replace('주차', '').astype(int)


# ========================================================
# 3. 처리하고 싶은 지역과 기상청 파일 경로 등록
# ========================================================
target_regions = {
    '강원': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/강원_기상청_데이터.csv',
    '경기': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경기_기상청_데이터.csv',
    '충북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충북_기상청_데이터.csv',
    '서울': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/서울_기상청_데이터.csv',
    '충남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충남_기상청_데이터.csv',
    '부산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/부산_기상청_데이터.csv',
    '대구': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대구_기상청_데이터.csv',
    '대전': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대전_기상청_데이터.csv',
    '전남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전남_기상청_데이터.csv',
    '전북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전북_기상청_데이터.csv',
    '경남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경남_기상청_데이터.csv',
    '경북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경북_기상청_데이터.csv',
    '제주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/제주_기상청_데이터.csv',
    '세종': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/세종_기상청_데이터.csv',
    '인천': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/인천_기상청_데이터.csv',
    '광주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/광주_기상청_데이터.csv',
    '울산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/울산_기상청_데이터.csv'
}


# 딕셔너리에 등록된 지역들을 하나씩 순회하며 "개별 파일"로 전처리 및 저장
for region_name, weather_path in target_regions.items():
    print(f"\n=========================================")
    print(f"[{region_name}] 모든 변수 시차 반영 및 소수점 정렬 시작")
    print(f"=========================================")

    # 해당 지역의 질병 데이터만 분리 필터링
    df_disease_region = df_disease_long[df_disease_long['지역'] == region_name].copy()

    # 날씨 데이터 로드
    try:
        df_w = pd.read_csv(weather_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_w = pd.read_csv(weather_path, encoding='cp949')

    if '일강수량(mm)' in df_w.columns:
        df_w['일강수량(mm)'] = pd.to_numeric(df_w['일강수량(mm)'], errors='coerce').fillna(0)

    df_w['일시'] = pd.to_datetime(df_w['일시'])
    iso_cal = df_w['일시'].dt.isocalendar()
    df_w['연度'] = iso_cal.year
    df_w['주차'] = iso_cal.week
    df_w['지역'] = region_name

    # 일별 지점 데이터 압축 (관측소가 여러 개일 경우 대비 일별 평균/최대/최소화)
    df_w_daily = df_w.groupby(['일시', '지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'mean', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 주간 단위 압축
    df_w_weekly = df_w_daily.groupby(['지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'sum', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 병합 전 시간 순서대로 확실하게 정렬 (Shift 연산용)
    df_w_weekly = df_w_weekly.sort_values(['연度', '주차']).reset_index(drop=True)

    # 모든 날씨 변수의 1주전, 2주전, 3주전 데이터 생성
    weather_cols = {
        '평균기온(°C)': '평균기온',
        '최고기온(°C)': '최고기온',
        '최저기온(°C)': '최저기온',
        '일강수량(mm)': '누적강수량',
        '평균 풍속(m/s)': '평균풍속',
        '평균 상대습도(%)': '평균상대습도'
    }

    # for origin_col, rename_col in weather_cols.items():
    #     for lag in [1, 2, 3]:
    #         df_w_weekly[f'{lag}주전_{rename_col}'] = df_w_weekly.groupby('지역')[origin_col].shift(lag)

    # ⭐⭐⭐ [핵심 수정] 루프 순서를 변경하여 주차별로 컬럼을 모아서 생성 ⭐⭐⭐
    # lag(1, 2, 3)을 바깥쪽 루프로 빼서 1주전 변수들이 먼저 쭉 생기고, 그다음 2주전 변수들이 쭉 생기게 만듭니다.
    for lag in [1, 2, 3]:
        for origin_col, rename_col in weather_cols.items():
            df_w_weekly[f'{lag}주전_{rename_col}'] = df_w_weekly.groupby('지역')[origin_col].shift(lag)

    # 컬럼명 매칭을 위해 이름 변경
    df_w_weekly.rename(columns={'연度': '연도'}, inplace=True)

    # 해당 지역의 질병 데이터와 날씨 테이블 병합
    df_region_merged = pd.merge(df_disease_region, df_w_weekly, on=['지역', '연도', '주차'], how='inner')

    # 시간 순서대로 예쁘게 정렬
    df_region_final = df_region_merged.sort_values(by=['연도', '주차']).reset_index(drop=True)

    # ⭐⭐⭐ [핵심 수정] 소수점이 있는 모든 수치형 컬럼을 소수점 둘째 자리로 반올림 ⭐⭐⭐
    # 환자수, 연도, 주차 같은 정수형을 제외한 실수형(float) 컬럼만 골라서 round(2) 적용
    float_cols = df_region_final.select_dtypes(include=[np.float64, float]).columns
    df_region_final[float_cols] = df_region_final[float_cols].round(2)

    # 💾 지역별 개별 파일로 저장
    output_path = f'/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/{region_name}_AI학습용_최종데이터.csv'
    df_region_final.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"▶ [{region_name}] 소수점 2자리 정리 완료된 데이터 확인:")
    print(df_region_final[['연도', '주차', '환자수', '평균기온(°C)', '1주전_평균기온', '2주전_평균풍속']].head(3))
    print(f"✅ 저장 완료: {output_path}")

print("\n🎉 모든 변수 반영 및 소수점 2자리 정리가 완료되었습니다!")

질병 데이터 로드를 시작합니다...

[강원] 모든 변수 시차 반영 및 소수점 정렬 시작
▶ [강원] 소수점 2자리 정리 완료된 데이터 확인:
     연도  주차  환자수  평균기온(°C)  1주전_평균기온  2주전_평균풍속
0  2016   1  1.0      2.66      4.47      2.54
1  2016   2  0.0      0.44      2.66      3.57
2  2016   3  1.0     -4.87      0.44      3.70
✅ 저장 완료: /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/강원_AI학습용_최종데이터.csv

[경기] 모든 변수 시차 반영 및 소수점 정렬 시작
▶ [경기] 소수점 2자리 정리 완료된 데이터 확인:
     연도  주차  환자수  평균기온(°C)  1주전_평균기온  2주전_평균풍속
0  2016   1  2.0     -1.37     -1.00      1.27
1  2016   2  1.0     -2.81     -1.37      0.93
2  2016   3  3.0     -9.37     -2.81      1.47
✅ 저장 완료: /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/경기_AI학습용_최종데이터.csv

[충북] 모든 변수 시차 반영 및 소수점 정렬 시작
▶ [충북] 소수점 2자리 정리 완료된 데이터 확인:
     연도  주차  환자수  평균기온(°C)  1주전_평균기온  2주전_평균풍속
0  2016   1  0.0      0.59      1.60      1.33
1  2016   2  0.0     -0.93      0.59      0.99
2  2016   3  0.0     -8.36     -0.93      1.37
✅ 저장 완료: /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/충북_AI학습용_최종데이터.csv

[서울] 모든 변수

In [16]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트 (필요 시 주석 해제)
# drive.mount('/content/drive')

# 2. 질병 데이터 로드 및 기본 전처리 (Wide -> Long)
disease_path = '/content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/merged_zzzzgamusi_data.csv'

print("질병 데이터 로드를 시작합니다...")
try:
    df_disease = pd.read_csv(disease_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_disease = pd.read_csv(disease_path, encoding='cp949')

if '합계' in df_disease.columns:
    df_disease = df_disease.drop(columns=['합계'])

# ⭐ 질병 데이터에서 '세종' 지역 완전히 제외
df_disease = df_disease[df_disease['지역'] != '세종'].reset_index(drop=True)

# 가로 데이터를 세로 데이터로 변환 (53주차 컬럼도 '53' 숫자로 완벽 변환)
df_disease_long = df_disease.melt(id_vars=['지역', '연도'], var_name='주차', value_name='환자수')
df_disease_long['주차'] = df_disease_long['주차'].str.replace('주차', '').astype(int)


# ========================================================
# 3. 처리하고 싶은 지역과 기상청 파일 경로 등록
# ========================================================
# 새로운 지역 파일이 생기면 아래 딕셔너리에 똑같은 형식으로 경로를 추가해 주시면 됩니다.
target_regions = {
    '강원': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/강원_기상청_데이터.csv',
    '경기': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경기_기상청_데이터.csv',
    '충북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충북_기상청_데이터.csv',
    '서울': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/서울_기상청_데이터.csv',
    '충남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충남_기상청_데이터.csv',
    '부산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/부산_기상청_데이터.csv',
    '대구': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대구_기상청_데이터.csv',
    '대전': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대전_기상청_데이터.csv',
    '전남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전남_기상청_데이터.csv',
    '전북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전북_기상청_데이터.csv',
    '경남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경남_기상청_데이터.csv',
    '경북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경북_기상청_데이터.csv',
    '제주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/제주_기상청_데이터.csv',
    '세종': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/세종_기상청_데이터.csv',
    '인천': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/인천_기상청_데이터.csv',
    '광주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/광주_기상청_데이터.csv',
    '울산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/울산_기상청_데이터.csv'
}


# 딕셔너리에 등록된 지역들을 하나씩 순회하며 "개별 파일"로 전처리 및 저장
for region_name, weather_path in target_regions.items():
    print(f"\n=========================================")
    print(f"[{region_name}] 53주차 보정 및 개별 파일 저장 시작")
    print(f"=========================================")

    # 3-1. 해당 지역의 질병 데이터만 분리 필터링
    df_disease_region = df_disease_long[df_disease_long['지역'] == region_name].copy()

    # 3-2. 날씨 데이터 로드
    try:
        df_w = pd.read_csv(weather_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_w = pd.read_csv(weather_path, encoding='cp949')

    if '일강수량(mm)' in df_w.columns:
        df_w['일강수량(mm)'] = pd.to_numeric(df_w['일강수량(mm)'], errors='coerce').fillna(0)

    df_w['일시'] = pd.to_datetime(df_w['일시'])
    iso_cal = df_w['일시'].dt.isocalendar()

    df_w['연度'] = iso_cal.year
    df_w['주차'] = iso_cal.week
    df_w['지역'] = region_name

    # ⭐⭐⭐ [핵심 수정: 개별 파일에도 53주차 살리기 로직 적용] ⭐⭐⭐
    # 2016년 12월 26일~31일 사이의 날씨 데이터를 질병 데이터와 짝이 맞게 '53주차'로 강제 보정합니다.
    is_2016_end = (df_w['일시'].dt.year == 2016) & (df_w['일시'].dt.month == 12) & (df_w['일시'].dt.day >= 26)
    df_w.loc[is_2016_end, '주차'] = 53
    df_w.loc[is_2016_end, '연度'] = 2016

    # 일별 지점 데이터 압축 (여러 관측소 평균/최대/최소화)
    df_w_daily = df_w.groupby(['일시', '지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'mean', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 주간 단위 압축
    df_w_weekly = df_w_daily.groupby(['지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'sum', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 병합 전 시간 순서대로 확실하게 정렬 (Shift 연산용)
    df_w_weekly = df_w_weekly.sort_values(['연度', '주차']).reset_index(drop=True)

    # 대상 날씨 변수 정의
    weather_cols = {
        '평균기온(°C)': '평균기온', '최고기온(°C)': '최고기온', '최저기온(°C)': '최저기온',
        '일강수량(mm)': '누적강수량', '평균 풍속(m/s)': '평균풍속', '평균 상대습도(%)': '평균상대습도'
    }

    # 주차별로 컬럼 모으기 (1주전 전체 -> 2주전 전체 -> 3주전 전체)
    for lag in [1, 2, 3]:
        for origin_col, rename_col in weather_cols.items():
            df_w_weekly[f'{lag}주전_{rename_col}'] = df_w_weekly.groupby('지역')[origin_col].shift(lag)

    # 컬럼명 매칭을 위해 이름 변경
    df_w_weekly.rename(columns={'연度': '연도'}, inplace=True)

    # 3-3. 해당 지역의 질병 데이터와 전처리된 날씨 데이터 병합 (53주차 완벽하게 살아남음)
    df_region_merged = pd.merge(df_disease_region, df_w_weekly, on=['지역', '연도', '주차'], how='inner')

    # 시간 순서대로 예쁘게 정렬
    df_region_final = df_region_merged.sort_values(by=['연도', '주차']).reset_index(drop=True)

    # 환자수가 빈칸(NaN)인 곳은 0으로 채우기
    df_region_final['환자수'] = df_region_final['환자수'].fillna(0)

    # 소수점이 있는 모든 수치형 컬럼을 소수점 둘째 자리로 반올림
    float_cols = df_region_final.select_dtypes(include=[np.float64, float]).columns
    df_region_final[float_cols] = df_region_final[float_cols].round(2)

    # 3-4. 💾 지역별 개별 파일로 최종 저장
    output_path = f'/content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/{region_name}_AI학습용_최종데이터.csv'
    df_region_final.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f"✅ [{region_name}] 53주차 포함 및 정렬 완료 파일 저장 완료:\n👉 {output_path}")

print("\n🎉 모든 요구사항(53주차 유지, 세종 제외, 주차별 정렬, 소수점 제한)이 반영된 지역별 개별 파일 생성이 완료되었습니다!")

질병 데이터 로드를 시작합니다...

[강원] 53주차 보정 및 개별 파일 저장 시작
✅ [강원] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/강원_AI학습용_최종데이터.csv

[경기] 53주차 보정 및 개별 파일 저장 시작
✅ [경기] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/경기_AI학습용_최종데이터.csv

[충북] 53주차 보정 및 개별 파일 저장 시작
✅ [충북] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/충북_AI학습용_최종데이터.csv

[서울] 53주차 보정 및 개별 파일 저장 시작
✅ [서울] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/서울_AI학습용_최종데이터.csv

[충남] 53주차 보정 및 개별 파일 저장 시작
✅ [충남] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/충남_AI학습용_최종데이터.csv

[부산] 53주차 보정 및 개별 파일 저장 시작
✅ [부산] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/부산_AI학습용_최종데이터.csv

[대구] 53주차 보정 및 개별 파일 저장 시작
✅ [대구] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/대구_AI학습용_최종데이터.csv

[대전] 53주차 보정 및 개별 파일 저장 시작
✅ [대전] 53주차 포함 및 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/학습용/대전_AI학습용_최종데이터.

In [7]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트 (필요 시 주석 해제)
# drive.mount('/content/drive')

# 2. 질병 데이터 로드 및 기본 전처리 (Wide -> Long)
disease_path = '/content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/merged_zzzzgamusi_data.csv'

print("질병 데이터 로드를 시작합니다...")
try:
    df_disease = pd.read_csv(disease_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_disease = pd.read_csv(disease_path, encoding='cp949')

if '합계' in df_disease.columns:
    df_disease = df_disease.drop(columns=['합계'])

# 가로 데이터를 세로 데이터로 변환 (전체 지역 포함)
df_disease_long = df_disease.melt(id_vars=['지역', '연도'], var_name='주차', value_name='환자수')
df_disease_long['주차'] = df_disease_long['주차'].str.replace('주차', '').astype(int)


# ========================================================
# 3. ⭐️ 통합하고 싶은 모든 지역과 기상청 파일 경로 등록
# ========================================================
# 여기에 가지고 계신 모든 지역의 파일 경로를 추가해 주시면 한 파일로 다 합쳐집니다!
target_regions = {
    '강원': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/강원_기상청_데이터.csv',
    '경기': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경기_기상청_데이터.csv',
    '충북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충북_기상청_데이터.csv',
    '서울': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/서울_기상청_데이터.csv',
    '충남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충남_기상청_데이터.csv',
    '부산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/부산_기상청_데이터.csv',
    '대구': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대구_기상청_데이터.csv',
    '대전': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대전_기상청_데이터.csv',
    '전남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전남_기상청_데이터.csv',
    '전북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전북_기상청_데이터.csv',
    '경남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경남_기상청_데이터.csv',
    '경북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경북_기상청_데이터.csv',
    '제주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/제주_기상청_데이터.csv',
    '세종': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/세종_기상청_데이터.csv',
    '인천': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/인천_기상청_데이터.csv',
    '광주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/광주_기상청_데이터.csv',
    '울산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/울산_기상청_데이터.csv'
}

all_weather_list = []

# 딕셔너리에 등록된 지역들을 순회하며 날씨 전처리 진행
for region_name, weather_path in target_regions.items():
    print(f"[{region_name}] 날씨 데이터 전처리 및 변환 중...")

    try:
        df_w = pd.read_csv(weather_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_w = pd.read_csv(weather_path, encoding='cp949')

    if '일강수량(mm)' in df_w.columns:
        df_w['일강수량(mm)'] = pd.to_numeric(df_w['일강수량(mm)'], errors='coerce').fillna(0)

    df_w['일시'] = pd.to_datetime(df_w['일시'])
    iso_cal = df_w['일시'].dt.isocalendar()
    df_w['연度'] = iso_cal.year
    df_w['주차'] = iso_cal.week
    df_w['지역'] = region_name

    # 일별 지점 데이터 압축
    df_w_daily = df_w.groupby(['일시', '지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'mean', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 주간 단위 압축
    df_w_weekly = df_w_daily.groupby(['지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'sum', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 시간 순 정렬 (Shift 연산용)
    df_w_weekly = df_w_weekly.sort_values(['연度', '주차']).reset_index(drop=True)

    # 대상 날씨 변수 정의
    weather_cols = {
        '평균기온(°C)': '평균기온', '최고기온(°C)': '최고기온', '최저기온(°C)': '최저기온',
        '일강수량(mm)': '누적강수량', '평균 풍속(m/s)': '평균풍속', '평균 상대습도(%)': '평균상대습도'
    }

    # 주차별로 컬럼 모으기 (1주전 전체 -> 2주전 전체 -> 3주전 전체)
    for lag in [1, 2, 3]:
        for origin_col, rename_col in weather_cols.items():
            df_w_weekly[f'{lag}주전_{rename_col}'] = df_w_weekly.groupby('지역')[origin_col].shift(lag)

    df_w_weekly.rename(columns={'연度': '연도'}, inplace=True)

    # 전처리가 끝난 개별 지역 날씨를 리스트에 임시 저장
    all_weather_list.append(df_w_weekly)

print("\n모든 지역의 날씨 데이터를 하나로 결합합니다...")
# 개별 날씨 테이블들을 세로로 길게 합치기
df_all_weather = pd.concat(all_weather_list, ignore_index=True)


# ========================================================
# 4. 최종 통합 데이터 병합 (전체 질병 데이터 + 통합 날씨 데이터)
# ========================================================
# 이 시점에는 원본 질병 CSV 파일에 들어있던 순서대로 지역이 배치됩니다.
df_final_merged = pd.merge(df_disease_long, df_all_weather, on=['지역', '연도', '주차'], how='inner')

# 소수점이 있는 모든 수치형 컬럼을 소수점 둘째 자리로 반올림
float_cols = df_final_merged.select_dtypes(include=[np.float64, float]).columns
df_final_merged[float_cols] = df_final_merged[float_cols].round(2)


# ========================================================
# 5. 💾 [버전 1] 원본 정렬 순서대로 저장 (검수용)
# ========================================================
output_path_raw = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv'
df_final_merged.to_csv(output_path_raw, index=False, encoding='utf-8-sig')
print(f"✅ [버전 1] 원본 순서 파일 저장 완료:\n👉 {output_path_raw}")


# ========================================================
# 6. 💾 [버전 2] 지역 -> 연도 -> 주차 순으로 예쁘게 정렬해서 저장 (AI 모델 학습용)
# ========================================================
df_sorted = df_final_merged.sort_values(by=['지역', '연도', '주차']).reset_index(drop=True)

output_path_sorted = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv'
df_sorted.to_csv(output_path_sorted, index=False, encoding='utf-8-sig')
print(f"✅ [버전 2] 정렬 완료 파일 저장 완료:\n👉 {output_path_sorted}")

print("\n🎉 전국 통합 데이터셋 빌드가 완벽하게 마무리되었습니다!")

질병 데이터 로드를 시작합니다...
[강원] 날씨 데이터 전처리 및 변환 중...
[경기] 날씨 데이터 전처리 및 변환 중...
[충북] 날씨 데이터 전처리 및 변환 중...
[서울] 날씨 데이터 전처리 및 변환 중...
[충남] 날씨 데이터 전처리 및 변환 중...
[부산] 날씨 데이터 전처리 및 변환 중...
[대구] 날씨 데이터 전처리 및 변환 중...
[대전] 날씨 데이터 전처리 및 변환 중...
[전남] 날씨 데이터 전처리 및 변환 중...
[전북] 날씨 데이터 전처리 및 변환 중...
[경남] 날씨 데이터 전처리 및 변환 중...
[경북] 날씨 데이터 전처리 및 변환 중...
[제주] 날씨 데이터 전처리 및 변환 중...
[세종] 날씨 데이터 전처리 및 변환 중...
[인천] 날씨 데이터 전처리 및 변환 중...
[광주] 날씨 데이터 전처리 및 변환 중...
[울산] 날씨 데이터 전처리 및 변환 중...

모든 지역의 날씨 데이터를 하나로 결합합니다...
✅ [버전 1] 원본 순서 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv
✅ [버전 2] 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv

🎉 전국 통합 데이터셋 빌드가 완벽하게 마무리되었습니다!


In [8]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트 (필요 시 주석 해제)
# drive.mount('/content/drive')

# 2. 질병 데이터 로드 및 기본 전처리 (Wide -> Long)
disease_path = '/content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/merged_zzzzgamusi_data.csv'

print("질병 데이터 로드를 시작합니다...")
try:
    df_disease = pd.read_csv(disease_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_disease = pd.read_csv(disease_path, encoding='cp949')

# 불필요한 '합계' 컬럼 제거
if '합계' in df_disease.columns:
    df_disease = df_disease.drop(columns=['합계'])

# ⭐⭐⭐ [핵심 수정] 질병 데이터에서 '세종' 지역 아예 빼버리기 ⭐⭐⭐
df_disease = df_disease[df_disease['지역'] != '세종'].reset_index(drop=True)

# 가로 데이터를 세로 데이터로 변환 (세종이 제외된 상태로 변환됨)
df_disease_long = df_disease.melt(id_vars=['지역', '연도'], var_name='주차', value_name='환자수')
df_disease_long['주차'] = df_disease_long['주차'].str.replace('주차', '').astype(int)


# ========================================================
# 3. 통합하고 싶은 모든 지역과 기상청 파일 경로 등록 (세종은 제외)
# ========================================================
target_regions = {
    '강원': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/강원_기상청_데이터.csv',
    '경기': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경기_기상청_데이터.csv',
    '충북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충북_기상청_데이터.csv',
    '서울': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/서울_기상청_데이터.csv',
    '충남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충남_기상청_데이터.csv',
    '부산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/부산_기상청_데이터.csv',
    '대구': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대구_기상청_데이터.csv',
    '대전': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대전_기상청_데이터.csv',
    '전남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전남_기상청_데이터.csv',
    '전북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전북_기상청_데이터.csv',
    '경남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경남_기상청_데이터.csv',
    '경북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경북_기상청_데이터.csv',
    '제주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/제주_기상청_데이터.csv',
    '세종': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/세종_기상청_데이터.csv',
    '인천': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/인천_기상청_데이터.csv',
    '광주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/광주_기상청_데이터.csv',
    '울산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/울산_기상청_데이터.csv'
}

all_weather_list = []

# 딕셔너리에 등록된 지역들을 순회하며 날씨 전처리 진행
for region_name, weather_path in target_regions.items():
    print(f"[{region_name}] 날씨 데이터 전처리 및 변환 중...")

    try:
        df_w = pd.read_csv(weather_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_w = pd.read_csv(weather_path, encoding='cp949')

    if '일강수량(mm)' in df_w.columns:
        df_w['일강수량(mm)'] = pd.to_numeric(df_w['일강수량(mm)'], errors='coerce').fillna(0)

    df_w['일시'] = pd.to_datetime(df_w['일시'])
    iso_cal = df_w['일시'].dt.isocalendar()
    df_w['연度'] = iso_cal.year
    df_w['주차'] = iso_cal.week
    df_w['지역'] = region_name

    # 일별 지점 데이터 압축
    df_w_daily = df_w.groupby(['일시', '지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'mean', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 주간 단위 압축
    df_w_weekly = df_w_daily.groupby(['지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'sum', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 시간 순 정렬 (Shift 연산용)
    df_w_weekly = df_w_weekly.sort_values(['연度', '주차']).reset_index(drop=True)

    # 대상 날씨 변수 정의
    weather_cols = {
        '평균기온(°C)': '평균기온', '최고기온(°C)': '최고기온', '최저기온(°C)': '최저기온',
        '일강수량(mm)': '누적강수량', '평균 풍속(m/s)': '평균풍속', '평균 상대습도(%)': '평균상대습도'
    }

    # 주차별로 컬럼 모으기 (1주전 전체 -> 2주전 전체 -> 3주전 전체)
    for lag in [1, 2, 3]:
        for origin_col, rename_col in weather_cols.items():
            df_w_weekly[f'{lag}주전_{rename_col}'] = df_w_weekly.groupby('지역')[origin_col].shift(lag)

    df_w_weekly.rename(columns={'연度': '연도'}, inplace=True)

    # 전처리가 끝난 개별 지역 날씨를 리스트에 임시 저장
    all_weather_list.append(df_w_weekly)

print("\n모든 지역의 날씨 데이터를 하나로 결합합니다...")
df_all_weather = pd.concat(all_weather_list, ignore_index=True)


# ========================================================
# 4. 최종 통합 데이터 병합 (전체 질병 데이터 + 통합 날씨 데이터)
# ========================================================
df_final_merged = pd.merge(df_disease_long, df_all_weather, on=['지역', '연도', '주차'], how='inner')

# 소수점이 있는 모든 수치형 컬럼을 소수점 둘째 자리로 반올림
float_cols = df_final_merged.select_dtypes(include=[np.float64, float]).columns
df_final_merged[float_cols] = df_final_merged[float_cols].round(2)


# ========================================================
# 5. 💾 [버전 1] 원본 정렬 순서대로 저장 (검수용)
# ========================================================
output_path_raw = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv'
df_final_merged.to_csv(output_path_raw, index=False, encoding='utf-8-sig')
print(f"✅ [버전 1] 원본 순서 파일 저장 완료:\n👉 {output_path_raw}")


# ========================================================
# 6. 💾 [버전 2] 지역 -> 연도 -> 주차 순으로 예쁘게 정렬해서 저장 (AI 모델 학습용)
# ========================================================
df_sorted = df_final_merged.sort_values(by=['지역', '연도', '주차']).reset_index(drop=True)

output_path_sorted = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv'
df_sorted.to_csv(output_path_sorted, index=False, encoding='utf-8-sig')
print(f"✅ [버전 2] 정렬 완료 파일 저장 완료:\n👉 {output_path_sorted}")

print("\n🎉 세종시가 완벽히 예외 처리된 전국 통합 데이터셋 빌드가 완료되었습니다!")

질병 데이터 로드를 시작합니다...
[강원] 날씨 데이터 전처리 및 변환 중...
[경기] 날씨 데이터 전처리 및 변환 중...
[충북] 날씨 데이터 전처리 및 변환 중...
[서울] 날씨 데이터 전처리 및 변환 중...
[충남] 날씨 데이터 전처리 및 변환 중...
[부산] 날씨 데이터 전처리 및 변환 중...
[대구] 날씨 데이터 전처리 및 변환 중...
[대전] 날씨 데이터 전처리 및 변환 중...
[전남] 날씨 데이터 전처리 및 변환 중...
[전북] 날씨 데이터 전처리 및 변환 중...
[경남] 날씨 데이터 전처리 및 변환 중...
[경북] 날씨 데이터 전처리 및 변환 중...
[제주] 날씨 데이터 전처리 및 변환 중...
[세종] 날씨 데이터 전처리 및 변환 중...
[인천] 날씨 데이터 전처리 및 변환 중...
[광주] 날씨 데이터 전처리 및 변환 중...
[울산] 날씨 데이터 전처리 및 변환 중...

모든 지역의 날씨 데이터를 하나로 결합합니다...
✅ [버전 1] 원본 순서 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv
✅ [버전 2] 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv

🎉 세종시가 완벽히 예외 처리된 전국 통합 데이터셋 빌드가 완료되었습니다!


In [14]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트 (필요 시 주석 해제)
# drive.mount('/content/drive')

# 2. 질병 데이터 로드 및 기본 전처리 (Wide -> Long)
disease_path = '/content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/merged_zzzzgamusi_data.csv'

print("질병 데이터 로드를 시작합니다...")
try:
    df_disease = pd.read_csv(disease_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_disease = pd.read_csv(disease_path, encoding='cp949')

if '합계' in df_disease.columns:
    df_disease = df_disease.drop(columns=['합계'])

# 질병 데이터에서 '세종' 지역 제외
df_disease = df_disease[df_disease['지역'] != '세종'].reset_index(drop=True)

# 가로 데이터를 세로 데이터로 변환 (이제 53주차 컬럼도 정상적으로 '53' 숫자로 변환됩니다)
df_disease_long = df_disease.melt(id_vars=['지역', '연도'], var_name='주차', value_name='환자수')
df_disease_long['주차'] = df_disease_long['주차'].str.replace('주차', '').astype(int)


# ========================================================
# 3. 통합하고 싶은 모든 지역과 기상청 파일 경로 등록
# ========================================================
target_regions = {
    '강원': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/강원_기상청_데이터.csv',
    '경기': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경기_기상청_데이터.csv',
    '충북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충북_기상청_데이터.csv',
    '서울': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/서울_기상청_데이터.csv',
    '충남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충남_기상청_데이터.csv',
    '부산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/부산_기상청_데이터.csv',
    '대구': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대구_기상청_데이터.csv',
    '대전': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대전_기상청_데이터.csv',
    '전남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전남_기상청_데이터.csv',
    '전북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전북_기상청_데이터.csv',
    '경남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경남_기상청_데이터.csv',
    '경북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경북_기상청_데이터.csv',
    '제주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/제주_기상청_데이터.csv',
    '세종': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/세종_기상청_데이터.csv',
    '인천': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/인천_기상청_데이터.csv',
    '광주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/광주_기상청_데이터.csv',
    '울산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/울산_기상청_데이터.csv'
}

all_weather_list = []

# 딕셔너리에 등록된 지역들을 순회하며 날씨 전처리 진행
for region_name, weather_path in target_regions.items():
    print(f"[{region_name}] 날씨 데이터 전처리 및 53주차 보정 중...")

    try:
        df_w = pd.read_csv(weather_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_w = pd.read_csv(weather_path, encoding='cp949')

    if '일강수량(mm)' in df_w.columns:
        df_w['일강수량(mm)'] = pd.to_numeric(df_w['일강수량(mm)'], errors='coerce').fillna(0)

    df_w['일시'] = pd.to_datetime(df_w['일시'])
    iso_cal = df_w['일시'].dt.isocalendar()

    df_w['연度'] = iso_cal.year
    df_w['주차'] = iso_cal.week
    df_w['지역'] = region_name

    # ⭐⭐⭐ [핵심 수정: 53주차 살리기 로직] ⭐⭐⭐
    # 실제 날짜의 연도가 2016년인데 표준 주차가 52주차이고, 12월 26일~31일 사이라면 강제로 '53주차'로 바꿉니다.
    # (질병 데이터의 2016년 53주차 포맷과 매칭시키기 위함)
    is_2016_end = (df_w['일시'].dt.year == 2016) & (df_w['일시'].dt.month == 12) & (df_w['일시'].dt.day >= 26)
    df_w.loc[is_2016_end, '주차'] = 53
    df_w.loc[is_2016_end, '연度'] = 2016  # ISO 주차가 2017년으로 넘어가는 것도 방지

    # 일별 지점 데이터 압축
    df_w_daily = df_w.groupby(['일시', '지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'mean', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 주간 단위 압축
    df_w_weekly = df_w_daily.groupby(['지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'sum', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 시간 순 정렬 (Shift 연산용)
    df_w_weekly = df_w_weekly.sort_values(['연度', '주차']).reset_index(drop=True)

    # 대상 날씨 변수 정의
    weather_cols = {
        '평균기온(°C)': '평균기온', '최고기온(°C)': '최고기온', '최저기온(°C)': '최저기온',
        '일강수량(mm)': '누적강수량', '평균 풍속(m/s)': '평균풍속', '평균 상대습도(%)': '평균상대습도'
    }

    # 주차별로 컬럼 모으기 (1주전 전체 -> 2주전 전체 -> 3주전 전체)
    for lag in [1, 2, 3]:
        for origin_col, rename_col in weather_cols.items():
            df_w_weekly[f'{lag}주전_{rename_col}'] = df_w_weekly.groupby('지역')[origin_col].shift(lag)

    df_w_weekly.rename(columns={'연度': '연도'}, inplace=True)

    # 리스트에 임시 저장
    all_weather_list.append(df_w_weekly)

print("\n모든 지역의 날씨 데이터를 하나로 결합합니다...")
df_all_weather = pd.concat(all_weather_list, ignore_index=True)


# ========================================================
# 4. 최종 통합 데이터 병합 (전체 질병 데이터 + 통합 날씨 데이터)
# ========================================================
# 이제 2016년 53주차 날씨가 생성되었으므로 inner join을 해도 53주차가 삭제되지 않고 살아남습니다!
df_final_merged = pd.merge(df_disease_long, df_all_weather, on=['지역', '연도', '주차'], how='inner')

# 소수점 둘째 자리 반올림
float_cols = df_final_merged.select_dtypes(include=[np.float64, float]).columns
df_final_merged[float_cols] = df_final_merged[float_cols].round(2)

# 환자수가 빈칸(NaN)인 곳은 0으로 채우기
df_region_final['환자수'] = df_region_final['환자수'].fillna(0)

# ========================================================
# 5. 💾 [버전 1] 원본 정렬 순서대로 저장 (검수용)
# ========================================================
output_path_raw = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv'
df_final_merged.to_csv(output_path_raw, index=False, encoding='utf-8-sig')
print(f"✅ [버전 1] 원본 순서 파일 저장 완료:\n👉 {output_path_raw}")


# ========================================================
# 6. 💾 [버전 2] 지역 -> 연도 -> 주차 순으로 예쁘게 정렬해서 저장 (AI 모델 학습용)
# ========================================================
df_sorted = df_final_merged.sort_values(by=['지역', '연도', '주차']).reset_index(drop=True)

output_path_sorted = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv'
df_sorted.to_csv(output_path_sorted, index=False, encoding='utf-8-sig')
print(f"✅ [버전 2] 정렬 완료 파일 저장 완료:\n👉 {output_path_sorted}")

print("\n🎉 2016년 53주차가 완벽히 반영된 전국 통합 데이터셋 빌드가 완료되었습니다!")

질병 데이터 로드를 시작합니다...
[강원] 날씨 데이터 전처리 및 53주차 보정 중...
[경기] 날씨 데이터 전처리 및 53주차 보정 중...
[충북] 날씨 데이터 전처리 및 53주차 보정 중...
[서울] 날씨 데이터 전처리 및 53주차 보정 중...
[충남] 날씨 데이터 전처리 및 53주차 보정 중...
[부산] 날씨 데이터 전처리 및 53주차 보정 중...
[대구] 날씨 데이터 전처리 및 53주차 보정 중...
[대전] 날씨 데이터 전처리 및 53주차 보정 중...
[전남] 날씨 데이터 전처리 및 53주차 보정 중...
[전북] 날씨 데이터 전처리 및 53주차 보정 중...
[경남] 날씨 데이터 전처리 및 53주차 보정 중...
[경북] 날씨 데이터 전처리 및 53주차 보정 중...
[제주] 날씨 데이터 전처리 및 53주차 보정 중...
[세종] 날씨 데이터 전처리 및 53주차 보정 중...
[인천] 날씨 데이터 전처리 및 53주차 보정 중...
[광주] 날씨 데이터 전처리 및 53주차 보정 중...
[울산] 날씨 데이터 전처리 및 53주차 보정 중...

모든 지역의 날씨 데이터를 하나로 결합합니다...
✅ [버전 1] 원본 순서 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv
✅ [버전 2] 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv

🎉 2016년 53주차가 완벽히 반영된 전국 통합 데이터셋 빌드가 완료되었습니다!


In [15]:
import pandas as pd
import numpy as np
from google.colab import drive

# 1. 구글 드라이브 마운트 (필요 시 주석 해제)
# drive.mount('/content/drive')

# 2. 질병 데이터 로드 및 기본 전처리 (Wide -> Long)
disease_path = '/content/drive/MyDrive/보충 프로젝트/쯔쯔가무시/merged_zzzzgamusi_data.csv'

print("질병 데이터 로드를 시작합니다...")
try:
    df_disease = pd.read_csv(disease_path, encoding='utf-8-sig')
except UnicodeDecodeError:
    df_disease = pd.read_csv(disease_path, encoding='cp949')

if '합계' in df_disease.columns:
    df_disease = df_disease.drop(columns=['합계'])

# 질병 데이터에서 '세종' 지역 제외
df_disease = df_disease[df_disease['지역'] != '세종'].reset_index(drop=True)

# 가로 데이터를 세로 데이터로 변환 (53주차 컬럼도 정상적으로 '53' 숫자로 변환됩니다)
df_disease_long = df_disease.melt(id_vars=['지역', '연도'], var_name='주차', value_name='환자수')
df_disease_long['주차'] = df_disease_long['주차'].str.replace('주차', '').astype(int)


# ========================================================
# 3. 통합하고 싶은 모든 지역과 기상청 파일 경로 등록 (세종 제외 완료)
# ========================================================
target_regions = {
    '강원': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/강원_기상청_데이터.csv',
    '경기': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경기_기상청_데이터.csv',
    '충북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충북_기상청_데이터.csv',
    '서울': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/서울_기상청_데이터.csv',
    '충남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/충남_기상청_데이터.csv',
    '부산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/부산_기상청_데이터.csv',
    '대구': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대구_기상청_데이터.csv',
    '대전': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/대전_기상청_데이터.csv',
    '전남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전남_기상청_데이터.csv',
    '전북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/전북_기상청_데이터.csv',
    '경남': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경남_기상청_데이터.csv',
    '경북': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/경북_기상청_데이터.csv',
    '제주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/제주_기상청_데이터.csv',
    '인천': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/인천_기상청_데이터.csv',
    '광주': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/광주_기상청_데이터.csv',
    '울산': '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/울산_기상청_데이터.csv'
}

all_weather_list = []

# 딕셔너리에 등록된 지역들을 하나씩 순회하며 날씨 전처리 진행
for region_name, weather_path in target_regions.items():
    print(f"[{region_name}] 날씨 데이터 전처리 및 53주차 보정 중...")

    try:
        df_w = pd.read_csv(weather_path, encoding='utf-8-sig')
    except UnicodeDecodeError:
        df_w = pd.read_csv(weather_path, encoding='cp949')

    if '일강수량(mm)' in df_w.columns:
        df_w['일강수량(mm)'] = pd.to_numeric(df_w['일강수량(mm)'], errors='coerce').fillna(0)

    df_w['일시'] = pd.to_datetime(df_w['일시'])
    iso_cal = df_w['일시'].dt.isocalendar()

    df_w['연度'] = iso_cal.year
    df_w['주차'] = iso_cal.week
    df_w['지역'] = region_name

    # ⭐ [53주차 살리기 로직]
    is_2016_end = (df_w['일시'].dt.year == 2016) & (df_w['일시'].dt.month == 12) & (df_w['일시'].dt.day >= 26)
    df_w.loc[is_2016_end, '주차'] = 53
    df_w.loc[is_2016_end, '연度'] = 2016

    # 일별 지점 데이터 압축
    df_w_daily = df_w.groupby(['일시', '지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'mean', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 주간 단위 압축
    df_w_weekly = df_w_daily.groupby(['지역', '연度', '주차']).agg({
        '평균기온(°C)': 'mean', '최고기온(°C)': 'max', '최저기온(°C)': 'min',
        '일강수량(mm)': 'sum', '평균 풍속(m/s)': 'mean', '평균 상대습도(%)': 'mean'
    }).reset_index()

    # 시간 순 정렬 (Shift 연산용)
    df_w_weekly = df_w_weekly.sort_values(['연度', '주차']).reset_index(drop=True)

    # 대상 날씨 변수 정의
    weather_cols = {
        '평균기온(°C)': '평균기온', '최고기온(°C)': '최고기온', '최저기온(°C)': '최저기온',
        '일강수량(mm)': '누적강수량', '평균 풍속(m/s)': '평균풍속', '평균 상대습도(%)': '평균상대습도'
    }

    # 주차별로 컬럼 모으기 (1주전 전체 -> 2주전 전체 -> 3주전 전체)
    for lag in [1, 2, 3]:
        for origin_col, rename_col in weather_cols.items():
            df_w_weekly[f'{lag}주전_{rename_col}'] = df_w_weekly.groupby('지역')[origin_col].shift(lag)

    df_w_weekly.rename(columns={'연度': '연도'}, inplace=True)

    # 리스트에 임시 저장
    all_weather_list.append(df_w_weekly)

print("\n모든 지역의 날씨 데이터를 하나로 결합합니다...")
df_all_weather = pd.concat(all_weather_list, ignore_index=True)


# ========================================================
# 4. 최종 통합 데이터 병합 (전체 질병 데이터 + 통합 날씨 데이터)
# ========================================================
df_final_merged = pd.merge(df_disease_long, df_all_weather, on=['지역', '연도', '주차'], how='inner')

# 소수점 둘째 자리 반올림
float_cols = df_final_merged.select_dtypes(include=[np.float64, float]).columns
df_final_merged[float_cols] = df_final_merged[float_cols].round(2)

# ⭐ [오류 수정 완료] df_region_final 대신 정확히 df_final_merged로 결측치 처리 진행
df_final_merged['환자수'] = df_final_merged['환자수'].fillna(0)


# ========================================================
# 5. 💾 [버전 1] 원본 정렬 순서대로 저장 (검수용)
# ========================================================
output_path_raw = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv'
df_final_merged.to_csv(output_path_raw, index=False, encoding='utf-8-sig')
print(f"✅ [버전 1] 원본 순서 파일 저장 완료:\n👉 {output_path_raw}")


# ========================================================
# 6. 💾 [버전 2] 지역 -> 연도 -> 주차 순으로 예쁘게 정렬해서 저장 (AI 모델 학습용)
# ========================================================
df_sorted = df_final_merged.sort_values(by=['지역', '연도', '주차']).reset_index(drop=True)

output_path_sorted = '/content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv'
df_sorted.to_csv(output_path_sorted, index=False, encoding='utf-8-sig')
print(f"✅ [버전 2] 정렬 완료 파일 저장 완료:\n👉 {output_path_sorted}")

print("\n🎉 2016년 53주차 및 2020년 53주차 결측치(0) 처리가 완벽히 반영된 통합 데이터셋이 빌드되었습니다!")

질병 데이터 로드를 시작합니다...
[강원] 날씨 데이터 전처리 및 53주차 보정 중...
[경기] 날씨 데이터 전처리 및 53주차 보정 중...
[충북] 날씨 데이터 전처리 및 53주차 보정 중...
[서울] 날씨 데이터 전처리 및 53주차 보정 중...
[충남] 날씨 데이터 전처리 및 53주차 보정 중...
[부산] 날씨 데이터 전처리 및 53주차 보정 중...
[대구] 날씨 데이터 전처리 및 53주차 보정 중...
[대전] 날씨 데이터 전처리 및 53주차 보정 중...
[전남] 날씨 데이터 전처리 및 53주차 보정 중...
[전북] 날씨 데이터 전처리 및 53주차 보정 중...
[경남] 날씨 데이터 전처리 및 53주차 보정 중...
[경북] 날씨 데이터 전처리 및 53주차 보정 중...
[제주] 날씨 데이터 전처리 및 53주차 보정 중...
[인천] 날씨 데이터 전처리 및 53주차 보정 중...
[광주] 날씨 데이터 전처리 및 53주차 보정 중...
[울산] 날씨 데이터 전처리 및 53주차 보정 중...

모든 지역의 날씨 데이터를 하나로 결합합니다...
✅ [버전 1] 원본 순서 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_통합데이터_원본순서.csv
✅ [버전 2] 정렬 완료 파일 저장 완료:
👉 /content/drive/MyDrive/보충 프로젝트/Data_전처리_완료/학습용/전국_AI학습용_최종데이터_정렬완료.csv

🎉 2016년 53주차 및 2020년 53주차 결측치(0) 처리가 완벽히 반영된 통합 데이터셋이 빌드되었습니다!
